In [17]:
import requests as rq
import pandas as pd

In [18]:
r = rq.get(
  url="https://earthquake.usgs.gov/fdsnws/event/1/query",
  params={
    "format": "geojson",
    "eventtype": "earthquake",
    "minmagnitude": "2.0",
    "starttime": "2026-10-01",
    "endtime": "2026-11-01", # To account for the daytime during Oct 31st
  },
)

print(f"response code: {r.status_code}")

response code: 200


In [19]:
df = pd.json_normalize(r.json()["features"])
df[["lon", "lat", "depth"]] = pd.DataFrame(df["geometry.coordinates"].tolist(), index=df.index)
df["time"] = pd.to_datetime(df["properties.time"], unit="ms")
df["region"] = df["properties.place"].str.split(", ").str[-1].fillna("Unknown")

srs = df.sample(n=100, random_state=42)
srs_var_mean = srs["properties.mag"].var(ddof=1) / len(srs)

def assign_stratum(reg):
  r_low = str(reg).lower()
  na = ["ca", "california", "ak", "alaska", "hi", "hawaii", "wa", "or", "nv", "ut", "id", "mt", "wy", "tx", "ok", "ks", "pr", "puerto rico", "mexico", "canada", "guatemala", "nicaragua", "costa rica", "panama", "el salvador", "dominican republic", "virgin islands"]
  asia = ["japan", "indonesia", "philippines", "taiwan", "china", "russia", "india", "nepal", "pakistan", "afghanistan", "iran", "turkey", "kuril", "kamchatka", "mariana", "bonin", "ryukyu", "timor", "myanmar"]
  europe = ["greece", "italy", "iceland", "spain", "portugal", "france", "romania", "albania", "croatia", "norway", "svalbard", "azores", "mediterranean"]
  
  if any(k in r_low for k in na):
    return "North America"
  elif any(k in r_low for k in asia):
    return "Asia"
  elif any(k in r_low for k in europe):
    return "Europe"
  else:
    return "South America / Oceania / Africa"

df["stratum"] = df["region"].apply(assign_stratum)
print(df["stratum"].value_counts())

stratum
North America                       2579
South America / Oceania / Africa     834
Asia                                 408
Europe                                32
Name: count, dtype: int64


In [20]:
conv = df.head(100)

for name, d in [("Population", df), ("Convenience", conv), ("SRS", srs)]:
  print(name, "| Mean Mag:", round(d["properties.mag"].mean(), 3), "| Span:", d["time"].max() - d["time"].min())

print("-" * 50)

n_total = 200
strata_counts = df["stratum"].value_counts()
allocations = (strata_counts / len(df) * n_total).round().astype(int)

allocations.iloc[0] += n_total - allocations.sum()

strat_sample = pd.concat([
  df[df["stratum"] == s].sample(n=count, random_state=42)
  for s, count in allocations.items()
])

srs_var_mean = srs["properties.mag"].var(ddof=1) / len(srs)

N = len(df)
strat_var_mean = 0
for s, n_h in allocations.items():
  W_h = strata_counts[s] / N
  s_h_sq = strat_sample[strat_sample["stratum"] == s]["properties.mag"].var(ddof=1)
  strat_var_mean += (W_h ** 2) * (s_h_sq / n_h)

print(f"Stratified Sample Size: {len(strat_sample)}")
print(f"SRS Mean Mag: {srs['properties.mag'].mean():.3f} | Variance of Mean: {srs_var_mean:.5f}")
print(f"Stratified Mean Mag: {strat_sample['properties.mag'].mean():.3f} | Variance of Mean: {strat_var_mean:.5f}")

Population | Mean Mag: 3.344 | Span: 30 days 23:57:33.409000
Convenience | Mean Mag: 3.0 | Span: 0 days 22:36:19.341000
SRS | Mean Mag: 3.264 | Span: 30 days 11:26:31.172000
--------------------------------------------------
Stratified Sample Size: 200
SRS Mean Mag: 3.264 | Variance of Mean: 0.01174
Stratified Mean Mag: 3.432 | Variance of Mean: 0.00437


## Question(s):
***Why does the convenience sample distort your estimate?***

By default, the USGS API returns results sorted in reverse chronological order (most recent earthquakes first). Taking the first 100 rows restricts the sample to a narrow window of time at the very end of October rather than the whole month.

Because earthquakes occur in clustered sequences (such as aftershock swarms following a major event), sampling only the most recent 100 rows captures whatever localized seismic activity was happening at that specific moment, skewing the average magnitude and dramatically shrinking the time span compared to the full population.

---

***Compare the variance of your stratified sample's mean magnitude with your simple random sample's.***

The variance of the stratified sample's mean magnitude is lower than the variance of the simple random sample's mean magnitude for two main reasons:

1. **Elimination of Between-Stratum Variance:** Different geographic regions have distinct seismic baselines (for example, USGS monitoring networks in North America detect many low-magnitude earthquakes, whereas remote oceanic or Asian subduction zones primarily record higher-magnitude events). Proportional stratification guarantees that every region is represented in exact proportion to the population, removing sampling error caused by over or under-sampling specific tectonic regions.
2. **Sample Size ($n = 200$ vs. $n = 100$):** Because the variance of a sample mean scales inversely with sample size ($\text{Var}(\bar{x}) \propto \frac{1}{n}$), doubling the sample size from $n = 100$ in the SRS to $n = 200$ in the stratified sample further cuts the sampling variance in half.